# PyTheranostics Data Ingestion

This example notebook shows two ways to bring DICOM data into PyTheranostics:

1. Auto-detection from a local directory (no network needed).
2. A built-in DICOM receiver that listens on a port and auto-organizes incoming data.

Pick the approach that matches your setup. You can run both independently.

In [ ]:
# Imports
from pathlib import Path
import pytheranostics as tx
from pytheranostics.imaging_ds.dicom_ingest import auto_setup_dosimetry_study, extract_patient_metadata

print('PyTheranostics version:', getattr(tx, '__version__', 'unknown'))

## Approach 1: Auto-detect from an existing directory

Point to a local folder containing your raw DICOM series (CT, SPECT/NM, and RTSTRUCT).
The helper will organize time points and extract injection/patient metadata automatically.

In [ ]:
# Set the base directory containing raw DICOM files (edit this)
base_dir = Path('/path/to/raw_dicom_folder')  # <-- change me

# Organize and extract metadata
study_info, ct_paths, spect_paths, rtstruct_files = auto_setup_dosimetry_study(
    base_dir=base_dir,
    patient_id=None,   # auto-detect from DICOM headers
    cleanup=True       # flatten single-child nested folders if any
)

# Quick summary
print('Patient ID:', study_info.get('patient_id'))
inj = study_info.get('injection_info', {})
print('Injection date:', inj.get('injection_date'), 'time:', inj.get('injection_time'))
print(f'CT time points:   {len(ct_paths)}')
print(f'SPECT time points: {len(spect_paths)}')
print(f'RTSTRUCT files:    {len(rtstruct_files)}')

# Example: show first CT/SPECT time point folders (if present)
print('First CT tp:', ct_paths[0] if ct_paths else None)
print('First SPECT tp:', spect_paths[0] if spect_paths else None)
print('First RTSTRUCT:', rtstruct_files[0] if rtstruct_files else None)

## Approach 2: Receive over the network and auto-organize

Start a DICOM receiver that accepts C-STORE, writes files to disk, and then auto-organizes
them into a PatientID/CycleX/tpY folder structure based on StudyDate and cycle gaps.

In [ ]:
# Configure and start the receiver (edit storage_root as needed)
from pytheranostics.dicomtools.dicom_receiver import create_receiver

storage_root = '/path/to/dicom_inbox'  # <-- change me; incoming DICOM will land here first
receiver = create_receiver(
    ae_title='PYTHERANOSTICS',
    port=11112,
    storage_root=storage_root,
    auto_organize=True,
    auto_organize_output_base=storage_root,   # organized output; defaults to storage_root
    auto_organize_cycle_gap_days=15,          # new cycle if >=15 days between study dates
    auto_organize_timepoint_separation_days=1,# separate timepoints by date
    auto_organize_debounce_seconds=120        # wait 2 minutes after last file per patient
)

receiver.start(blocking=False)
print('DICOM Receiver running on port 11112 (AE_TITLE=PYTHERANOSTICS)')
print('Tip: Point your PACS/modality to this AE Title and port on this machine.')

In [ ]:
# Optional: Health check (C-ECHO)
from pynetdicom import AE
from pynetdicom.sop_class import Verification

ae = AE(ae_title='TX_DOCS_TEST')
ae.add_requested_context(Verification)
assoc = ae.associate('127.0.0.1', 11112, ae_title=b'PYTHERANOSTICS')
print('Association established:', getattr(assoc, 'is_established', False))
if getattr(assoc, 'is_established', False):
    status = assoc.send_c_echo()
    print('C-ECHO status:', hex(status.Status) if status else None)
    assoc.release()

### Where to find the organized data

After reception, files are moved under: `storage_root/PatientID/CycleX/tpY/<Modality>` with RTSTRUCT under `CT/RTstruct`.
Timepoints are grouped by StudyDate; cycles split when gaps between dates are ≥ the configured threshold (default 15 days).

In [ ]:
# List organized cycles/timepoints for a patient (replace with your patient ID)
from pathlib import Path
patient_id = 'YOUR_PATIENT_ID'   # <-- change me
patient_root = Path(storage_root) / patient_id
print('Patient root:', patient_root)

if patient_root.exists():
    for cycle_dir in sorted(patient_root.glob('Cycle*')):
        print(' ', cycle_dir.name)
        for tp_dir in sorted(cycle_dir.glob('tp*')):
            print('    ', tp_dir.name, 'contents:', [p.name for p in tp_dir.iterdir() if p.is_dir()])
else:
    print('No organized data found yet. Send data to the receiver and try again.')

## Clean up (optional)

Stop the background receiver when you're done.

In [ ]:
try:
    receiver.stop()
    print('Receiver stopped')
except NameError:
    print('Receiver variable not defined in this session')

## Next steps

Use the organized `ct_paths`, `spect_paths`, and `rtstruct_files` from Approach 1 or the folders created by Approach 2 to build longitudinal studies and proceed with dosimetry.
Refer to the project documentation for examples creating `LongStudy` objects and running organ/voxel dosimetry.